In [1]:
!pip install -q transformers torch huggingface_hub

In [2]:
from huggingface_hub import login

# Paste your Hugging Face access token when prompted
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Hugging Face model repository
model_name = "SandipanMondal06/bert-routing-agent"  # Replace with your actual repo name

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"✅ Model loaded successfully on {device}.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded successfully on cpu.


In [4]:
# Access label mappings
id2label = model.config.id2label
label2id = model.config.label2id

print("Label Mapping:", id2label)

Label Mapping: {0: 'ACCOUNT', 1: 'CANCEL', 2: 'CONTACT', 3: 'DELIVERY', 4: 'FEEDBACK', 5: 'INVOICE', 6: 'ORDER', 7: 'PAYMENT', 8: 'REFUND', 9: 'SHIPPING', 10: 'SUBSCRIPTION'}


In [5]:
import torch

def predict_department(query):
    # Tokenize input
    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    # Perform inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    probabilities = torch.softmax(outputs.logits, dim=1)

    # Get predicted class and confidence
    confidence, predicted_class_id = torch.max(probabilities, dim=1)

    predicted_label = id2label[predicted_class_id.item()]

    return predicted_label, confidence.item()

In [6]:
# Example queries
queries = [
    "I want to cancel my order.",
    "Where is my refund?",
    "How can I update my account details?",
    "I did not receive my package."
]

for query in queries:
    label, confidence = predict_department(query)
    print(f"\nQuery: {query}")
    print(f"Predicted Department: {label}")
    print(f"Confidence: {confidence:.4f}")


Query: I want to cancel my order.
Predicted Department: ORDER
Confidence: 0.9995

Query: Where is my refund?
Predicted Department: REFUND
Confidence: 0.9997

Query: How can I update my account details?
Predicted Department: ACCOUNT
Confidence: 0.9998

Query: I did not receive my package.
Predicted Department: DELIVERY
Confidence: 0.9670
